# 02 — Colab runner (GPU training)

This notebook only **launches** code from the repository: all logic lives in `src/brain_tumor/`.

**Runtime → Change runtime type → GPU (T4)**, then *Run all*.

Colab secrets (key icon in the left bar, enable *Notebook access* for each):

| Secret | Needed? | Used for |
|---|---|---|
| `GITHUB_TOKEN` | **yes** while the repository is private | cloning the repo (and pushing tuned parameters) |
| `DAGSHUB_USER`, `DAGSHUB_TOKEN` | optional | remote MLflow server + `dvc pull` from DagsHub |

Without the DagsHub secrets the notebook still works: it rebuilds the dataset from Figshare with
`dvc repro` (~5 min), logs MLflow runs to a local `mlflow.db`, and the last section copies the
checkpoints, reports and MLflow database to your Google Drive so nothing is lost when the
session ends.

In [ ]:
# ---- Settings ----
GITHUB_USER = "Yassminefeki"
REPO = "brain-tumor-mlops"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/brain-tumor-mlops-results"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Credentials

In [ ]:
import os
from google.colab import userdata


def secret(name):
    try:
        return userdata.get(name) or None
    except Exception:  # secret missing or notebook access not granted
        return None


GITHUB_TOKEN = secret("GITHUB_TOKEN")
DAGSHUB_USER, DAGSHUB_TOKEN = secret("DAGSHUB_USER"), secret("DAGSHUB_TOKEN")
USE_DAGSHUB = bool(DAGSHUB_USER and DAGSHUB_TOKEN)

if USE_DAGSHUB:
    os.environ.update({
        "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USER}/{REPO}.mlflow",
        "MLFLOW_TRACKING_USERNAME": DAGSHUB_USER,
        "MLFLOW_TRACKING_PASSWORD": DAGSHUB_TOKEN,
    })
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
print("GitHub token:", "yes" if GITHUB_TOKEN else "no (public repo only)")
print("MLflow:", "DagsHub" if USE_DAGSHUB else "local mlflow.db (saved to Drive at the end)")

## 2. Get the code and install it

In [ ]:
auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
if not os.path.exists(f"/content/{REPO}"):
    !git clone -q -b {BRANCH} https://{auth}github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!pip install -q -e .
!git log -1 --oneline

## 3. Data

With DagsHub: pull the exact versioned data (~300 MB). Without it: rebuild it from Figshare;
`dvc.lock` in the repository guarantees the result is the same data version.

In [ ]:
remotes = !dvc remote list
has_remote = USE_DAGSHUB and any(line.startswith("origin") for line in remotes)
if has_remote:
    !dvc remote modify origin --local auth basic
    !dvc remote modify origin --local user {DAGSHUB_USER}
    !dvc remote modify origin --local password {DAGSHUB_TOKEN}
    !dvc pull data/processed data/splits.csv data/metadata.csv
else:
    !dvc repro
!python -m brain_tumor.data.validate

## 4. Experiments

Run the cells you need. Every run is logged to MLflow with its parameters, git commit,
data version, curves, confusion matrix, Grad-CAM and checkpoint.

In [ ]:
# 0. Majority-class baseline
!python -m brain_tumor.baseline

In [ ]:
# 1-2. Custom CNN without / with augmentation
!python -m brain_tumor.train train.model=custom_cnn train.augment=false
!python -m brain_tumor.train train.model=custom_cnn

In [ ]:
# 3-4. Pretrained models (two-stage fine-tuning)
!python -m brain_tumor.train train.model=resnet18
!python -m brain_tumor.train train.model=efficientnet_b0

In [ ]:
# 5. Leakage demonstration: same model, image-level split
!python -m brain_tumor.train train.model=resnet18 train.split_column=image_split

## 5. (Optional) Optuna tuning — same budget for every compared model

In [ ]:
!python -m brain_tumor.tune train.model=custom_cnn tune.n_trials=20
!python -m brain_tumor.train train.model=custom_cnn --best-params configs/best_params/custom_cnn.yaml --run-name custom_cnn-tuned

In [ ]:
# Push the tuned parameters back to GitHub (needs the GITHUB_TOKEN secret)
!git config user.email "colab@users.noreply.github.com" && git config user.name "Colab runner"
!git add configs/best_params && git commit -m "exp: tuned hyperparameters" || echo "nothing to commit"
!git push -q https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:{BRANCH}

## 6. (Optional) Bonus: 5-fold patient-level cross-validation

In [ ]:
for fold in range(5):
    !python -m brain_tumor.train train.model=resnet18 train.fold={fold}

## 7. Final model

Retrain the best model of section 4 (change `train.model` if it is not ResNet-18), register it
in the MLflow Model Registry and copy its checkpoint to `models/best.pt` for the API.

In [ ]:
!python -m brain_tumor.train train.model=resnet18 --register --promote --run-name resnet18-final

## 8. Save the results to Google Drive

Copies `models/` (checkpoints, curves, confusion matrices, Grad-CAM), `reports/`, `configs/`
and the local MLflow database. On your laptop, put `models/best.pt` in the project's `models/`
folder to serve it with `make api`, and copy `mlflow.db` to browse the runs with `mlflow ui`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
!cp -r models reports configs {DRIVE_DIR}/
!test -f mlflow.db && cp mlflow.db {DRIVE_DIR}/ || true
!ls -R {DRIVE_DIR} | head -40